# Reranking RAG with AI Wiki

This notebook implements a reranking-based Retrieval-Augmented Generation (RAG) pipeline on the Sci-Wiki dataset to evaluate how improved retrieval quality affects downstream answer generation. The pipeline first retrieves candidate passages using dense embeddings, then applies a cross-encoder reranker to reorder them based on relevance to the query before passing the top contexts to the language model. 

By using the AI Wiki dataset, which covers diverse concepts in artificial intelligence and often requires cross-topic understanding, this notebook aims to assess whether reranking helps the system retrieve more relevant context for answering conceptually complex questions when dealing with questions that require relational unerstanding.

---

**Getting Started**
1. Run pip install cells, and restart runtime if required.
2. Run the notebook cells as outlined below

In [1]:
# If needed, install deps
%pip install numpy sentence-transformers faiss-cpu pandas tqdm transformers
%pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 34.8 MB/s eta 0:00:00


In [2]:
from dataclasses import dataclass
from typing import List
from pathlib import Path
import json
import time
import os
import cohere

import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from transformers import pipeline




# Importing our Dataset

Here, we import the AI Wiki dataset, which was obtained by scraping wikipedia for AI related articles.

We also take a look at some samples of our dataset to help us better understand what our dataset looks like.

In [3]:
DATA_PATH = Path("../datasets/wiki_corpus.json")

docs = []
bad_rows = 0

with DATA_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

for i, item in enumerate(raw):
    title = str(item.get("title", "")).strip()
    text = str(item.get("content") or item.get("text") or "").strip()

    if not text:
        bad_rows += 1
        continue

    if not title:
        title = f"Wiki Doc {i}"

    url = item.get("url") or f"wiki_corpus#{title.replace(' ', '_')}"

    docs.append({
        "title": title,
        "url": url,
        "text": text,
    })

N_DOCS = None
if N_DOCS is not None:
    docs = docs[:N_DOCS]

print("Docs loaded:", len(docs))
print("Bad rows skipped:", bad_rows)
print("Sample keys:", docs[0].keys() if docs else "No docs loaded")


Docs loaded: 100
Bad rows skipped: 0
Sample keys: dict_keys(['title', 'url', 'text'])


Preprocessing:

- Removed Rows with empty text
- Normalized tabs/newlines/multiple spaces into clean single-spacing
- Stripped whitespace from title, url, and text


In [4]:
# Inspect + light preprocessing 
import re
from collections import Counter
import random

def clean_text(text):
    text = str(text)
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_docs = []
dropped_docs = 0

for d in docs:
    title = str(d.get("title", "")).strip()
    url = str(d.get("url", "")).strip()
    text = clean_text(d.get("text", ""))

    # Drop unusable rows
    if not text:
        dropped_docs += 1
        continue

    clean_docs.append({
        "title": title,
        "url": url,
        "text": text
    })

docs = clean_docs

# Basic dataset stats
text_lengths_chars = [len(d["text"]) for d in docs]
text_lengths_words = [len(d["text"].split()) for d in docs]
title_counts = Counter(d["title"] for d in docs if d["title"])
duplicate_titles = sum(1 for c in title_counts.values() if c > 1)

print("Usable docs:", len(docs))
print("Dropped docs:", dropped_docs)
print("Duplicate titles:", duplicate_titles)

if docs:
    ex = docs[0]
    preview = ex["text"][:300]
    preview_safe = preview.encode("ascii", "replace").decode("ascii")

    print("\n--- Sample document ---")
    print("Title:", ex["title"])
    print("URL:", ex["url"])
    print("Text length (chars):", len(ex["text"]))
    print("Approx words:", len(ex["text"].split()))
    print("Text snippet:", preview_safe, "...")

    print("\n--- Length summary ---")
    print(f"Chars  | min={min(text_lengths_chars)}, max={max(text_lengths_chars)}, avg={sum(text_lengths_chars)/len(text_lengths_chars):.1f}")
    print(f"Words  | min={min(text_lengths_words)}, max={max(text_lengths_words)}, avg={sum(text_lengths_words)/len(text_lengths_words):.1f}")

else:
    print("No usable documents found.")


Usable docs: 100
Dropped docs: 0
Duplicate titles: 4

--- Sample document ---
Title: Artificial intelligence
URL: wiki_corpus#Artificial_intelligence
Text length (chars): 84209
Approx words: 12811
Text snippet: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develo ...

--- Length summary ---
Chars  | min=207, max=84209, avg=19371.0
Words  | min=29, max=12811, avg=3001.6


# Chunking 

We perform chunking on our corpus below, with each chunk size being 180, with an overlap of 40.

In [5]:
# Chunk documents into passages 
from dataclasses import dataclass
from typing import List
from tqdm import tqdm

@dataclass
class Passage:
    doc_id: int
    chunk_id: int
    title: str
    url: str
    text: str
    n_words: int

def chunk_text(text: str, chunk_size: int = 180, overlap: int = 40) -> List[str]:
    words = text.split()
    chunks = []

    if not words:
        return chunks

    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks

passages = []

for doc_id, d in enumerate(tqdm(docs, desc="Chunking docs")):
    title = d["title"]
    url = d["url"]
    text = d["text"]

    chunks = chunk_text(text, chunk_size=180, overlap=40)

    for chunk_id, chunk in enumerate(chunks):
        n_words = len(chunk.split())

        # Skip very short chunks
        if n_words < 30:
            continue

        # Include title in retrieval text
        chunk_text_for_retrieval = f"{title}\n\n{chunk}" if title else chunk

        passages.append(
            Passage(
                doc_id=doc_id,
                chunk_id=chunk_id,
                title=title,
                url=url,
                text=chunk_text_for_retrieval,
                n_words=n_words
            )
        )

print("Total passages:", len(passages))

if passages:
    print("\n--- Sample passage ---")
    print("Doc ID:", passages[0].doc_id)
    print("Chunk ID:", passages[0].chunk_id)
    print("Title:", passages[0].title)
    print("URL:", passages[0].url)
    print("Words:", passages[0].n_words)
    print("Text preview:", passages[0].text[:400], "...")

Chunking docs: 100%|██████████| 100/100 [00:00<00:00, 1320.02it/s]

Total passages: 2164

--- Sample passage ---
Doc ID: 0
Chunk ID: 0
Title: Artificial intelligence
URL: wiki_corpus#Artificial_intelligence
Words: 180
Text preview: Artificial intelligence

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their  ...


# Convert chunks to embeddings

We convert our chunks to embeddings, with the model used in this process being all-MiniLM-L6-v2.

In [6]:
# Cache passages + embeddings 
CACHE_DIR = Path("cache_wiki_ref")
CACHE_DIR.mkdir(exist_ok=True)

PASSAGES_PATH = CACHE_DIR / "passages.jsonl"
EMB_PATH = CACHE_DIR / "embeddings.npy"

def save_passages(passages):
    with open(PASSAGES_PATH, "w", encoding="utf-8") as f:
        for p in passages:
            f.write(json.dumps(p.__dict__, ensure_ascii=True) + "\n")

def load_passages():
    out = []
    with open(PASSAGES_PATH, "r", encoding="utf-8") as f:
        for line in f:
            out.append(Passage(**json.loads(line)))
    return out

passages_loaded_from_cache = False
embeddings = None

if PASSAGES_PATH.exists() and EMB_PATH.exists():
    try:
        print("Loading cached passages + embeddings...")
        passages = load_passages()
        embeddings = np.load(EMB_PATH).astype("float32")

        if len(passages) != embeddings.shape[0]:
            raise ValueError(
                f"Cache mismatch: {len(passages)} passages but {embeddings.shape[0]} embeddings"
            )

        passages_loaded_from_cache = True
        print(f"Loaded {len(passages)} passages")
        print(f"Embedding shape: {embeddings.shape}")

    except Exception as e:
        print(f"Cache load failed: {e}")
        print("Will recompute passages/embeddings.")
        embeddings = None
else:
    print("No cache found; will compute embeddings.")

No cache found; will compute embeddings.


In [7]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
if embeddings is None:
    embeddings = embed_model.encode(
        [p.text for p in passages],
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')
    save_passages(passages)
    np.save(EMB_PATH, embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print('FAISS index ready')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

FAISS index ready


**Here, we set our reranker to use ms-marco-MiniLM-L-6-v2**

In [8]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def retrieve(query: str, top_k: int = 15):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, top_k)
    return scores[0], idxs[0]


def rerank(query: str, idxs):
    cands = [(i, passages[i]) for i in idxs if i != -1]
    pairs = [(query, p.text) for i, p in cands]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)
    return ranked  # list of (score, (idx, passage))


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

**Testing of single query on our current pipeline**

In [9]:
# Try a query from the test set or a custom one
TEST_PATH = Path('../test_questions_wiki.json')
with TEST_PATH.open('r', encoding='utf-8') as f:
    test_questions = json.load(f)

query = test_questions[0]['question']
print('Query:', query)

_, idxs = retrieve(query, top_k=15)
ranked = rerank(query, idxs)[:5]

for score, (idx, p) in ranked:
    snippet = p.text[:200].replace('\n', ' ')
    snippet_safe = snippet.encode('ascii', 'replace').decode('ascii')
    print(f'- score={score:.4f} | url={p.url}')
    print(f'  {snippet_safe}...')


Query: How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?
- score=8.4357 | url=wiki_corpus#Criss-cross_algorithm
  Criss-cross algorithm  algorithm was published independently by Tamas Terlaky and by Zhe-Min Wang; related algorithms appeared in unpublished reports by other authors. == Comparison with the simplex a...
- score=7.1934 | url=wiki_corpus#Criss-cross_algorithm
  Criss-cross algorithm  In mathematical optimization, the criss-cross algorithm is any of a family of algorithms for linear programming. Variants of the criss-cross algorithm also solve more general pr...
- score=6.9364 | url=wiki_corpus#Criss-cross_algorithm
  Criss-cross algorithm  of a (perturbed) cube in dimension D, according to a paper of Roos; Roos's paper modifies the Klee?Minty construction of a cube on which the simplex algorithm takes 2D steps. Li...
- score=6.6784 | url=wiki_corpus#Criss-cross_algorithm
  Criss-cross algorit

# Setup of LLM to answer our query

We use Cohere as our LLM to answer our questions, and pass into the context the chunks selected after reranking.

In [10]:
# Cohere helper for retryable chat calls 
# set your API key here directly. or use secrets
COHERE_API_KEY = "COHERE_API_KEY"



if not COHERE_API_KEY:
    raise ValueError("Cohere API key not found. Set COHERE_API_KEY first.")

co = cohere.ClientV2(api_key=COHERE_API_KEY)

def chat_with_retry(model: str, message: str, temperature: float = 0, max_retries: int = 3):
    """
    Retry wrapper for Cohere chat API.
    Returns the raw Cohere response object.
    """
    last_error = None

    for attempt in range(max_retries):
        try:
            response = co.chat(
                model=model,
                messages=[
                    {"role": "user", "content": message}
                ],
                temperature=temperature
            )
            return response

        except Exception as e:
            last_error = e
            wait = 2 * (attempt + 1)
            print(f"Chat failed (attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(wait)

    raise last_error

In [11]:
# ── Answer generation + full reranking RAG pipeline ─────────────────────────
def generate_answer(question, context, model="command-r7b-12-2024"):
    prompt = f"""Answer the question using only the context below.

If the context is insufficient, still provide the best possible answer based only on the context.
Do not use outside knowledge.
Be concise but complete.

Context:
{context}

Question:
{question}

Answer:"""

    response = chat_with_retry(
        model=model,
        message=prompt,
        temperature=0
    )
    return response.message.content[0].text.strip()


def build_context_from_ranked(ranked, max_passages=5):
    selected = ranked[:max_passages]
    context_parts = []

    for rank, (score, (idx, p)) in enumerate(selected, 1):
        context_parts.append(f"[Passage {rank}] Title: {p.title}\n{p.text}")

    return "\n\n".join(context_parts)


def reranking_rag(question, return_context=False, retrieve_k=15, rerank_k=5):
    # Dense retrieval
    _, idxs = retrieve(question, top_k=retrieve_k)

    # Rerank retrieved candidates
    ranked = rerank(question, idxs)

    # Keep only top reranked passages
    ranked = ranked[:rerank_k]

    # Build context from top reranked passages
    context = build_context_from_ranked(ranked, max_passages=rerank_k)

    # Generate final answer
    answer = generate_answer(question, context)

    if return_context:
        return {
            "answer": answer,
            "context": context,
            "ranked": ranked
        }

    return {"answer": answer}

In [17]:
# ── Sample test for reranking RAG ─────────────────────────────────────────────
test_question = "How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?"

test_result = reranking_rag(test_question, return_context=True, retrieve_k=15, rerank_k=5)

print("QUESTION:")
print(test_question)

print("\nANSWER:")
print(test_result["answer"])

print("\nCONTEXT PREVIEW:")
print(test_result["context"][:1200], "...")

QUESTION:
How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?

ANSWER:
The criss-cross algorithm differs from the simplex algorithm in that it is simpler, with only one phase, and its pivoting rules are based on the signs of coefficients rather than their real-number ordering. Unlike the simplex algorithm, which first finds a primal-feasible basis and then pivots to improve the objective function, the criss-cross algorithm directly selects entering and leaving variables based on the signs of coefficients, making it a purely combinatorial approach.

CONTEXT PREVIEW:
[Passage 1] Title: Criss-cross algorithm
Criss-cross algorithm

algorithm was published independently by Tamas Terlaky and by Zhe-Min Wang; related algorithms appeared in unpublished reports by other authors. == Comparison with the simplex algorithm for linear optimization == In linear programming, the criss-cross algorithm pivots between a seq

## Evaluation (LLM judge on test_questions_wiki)

Here, we use an LLM judge to judge the answers to the query using 3 metrics:
- **faithfulness**
- **answer_relevance**
- **context_relevance**


In [13]:
# LLM Judge
# Scores a single (question, context, answer) triple on 3 metrics.

def llm_judge(question, context, answer_text):
    """Score an answer using an LLM judge.

    Returns a dict with scores (1–5) for:
        faithfulness      – every claim is supported by the context
        answer_relevance  – answer directly addresses the question
        context_relevance – retrieved context was useful for the question

    A reasoning string is also returned for transparency.
    """
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.

Score the answer below on three criteria using a 1–5 integer scale:

  1 = very poor  |  3 = acceptable  |  5 = excellent

Criteria:
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context if context else "(no context retrieved)"}

ANSWER:
{answer_text}

Return ONLY a valid JSON object (no extra text):
{{
  "faithfulness": <1-5>,
  "answer_relevance": <1-5>,
  "context_relevance": <1-5>,
  "reasoning": "<one sentence explaining the scores>"
}}
"""
    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0
    )

    raw = response.message.content[0].text.strip()

    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()

    try:
        scores = json.loads(raw)

        for key in ("faithfulness", "answer_relevance", "context_relevance"):
            scores[key] = max(1, min(5, int(scores.get(key, 1))))

        return scores

    except (json.JSONDecodeError, ValueError):
        return {
            "faithfulness": 0,
            "answer_relevance": 0,
            "context_relevance": 0,
            "reasoning": "parse error"
        }

# Quick smoke test
_test = llm_judge(
    "What is machine learning?",
    "Machine learning -[is part of]-> Artificial intelligence",
    "Machine learning is a subfield of artificial intelligence."
)
print("Judge smoke test:", _test)

Judge smoke test: {'faithfulness': 5, 'answer_relevance': 5, 'context_relevance': 5, 'reasoning': 'The answer is accurate, directly addressing the question and supported by the context.'}


Now evaluation will be run on our full dataset.

In [14]:
# ── Step: Load evaluation questions ─────────────────────────────────────────
TEST_Q_PATH = Path("../test_questions_wiki.json")

with open(TEST_Q_PATH, "r", encoding="utf-8") as f:
    test_questions = json.load(f)

print("Loaded test questions:", len(test_questions))
print("Sample question:", test_questions[0]["question"])

Loaded test questions: 24
Sample question: How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?


In [15]:
DATASET_NAME = "wiki"

# ── Step: Run evaluation on all questions ───────────────────────────────────
EVAL_RESULTS_PATH = f"../datasets/eval_results_{DATASET_NAME}.json"
Path("data").mkdir(exist_ok=True)

# Always start fresh
eval_results = []
print("Running full evaluation from scratch (no resume)...")

for i, item in enumerate(test_questions):
    question = item["question"]

    print(f"[{i+1}/{len(test_questions)}] Evaluating: {question[:90]}")

    for attempt in range(3):
        try:
            result = reranking_rag(question, return_context=True, retrieve_k=15, rerank_k=5)
            ans = result["answer"]
            ctx = result["context"]

            scores = llm_judge(question, ctx, ans)
            break

        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  Attempt {attempt+1}/3 failed: {e}")
            time.sleep(wait)
    else:
        print("  Skipping after 3 failed attempts.")
        continue

    eval_results.append({
        "question": question,
        "reference_answer": item.get("reference_answer") or item.get("reference", ""),
        "reranking_rag_answer": ans,
        "context": ctx,
        "faithfulness": scores["faithfulness"],
        "answer_relevance": scores["answer_relevance"],
        "context_relevance": scores["context_relevance"],
        "reasoning": scores.get("reasoning", "")
    })

    # Save after every question
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    time.sleep(1)

print(f"\nDone. {len(eval_results)}/{len(test_questions)} results saved to {EVAL_RESULTS_PATH}")

Running full evaluation from scratch (no resume)...
[1/24] Evaluating: How does the criss-cross algorithm differ from the simplex algorithm in terms of its appro
[2/24] Evaluating: What is the significance of the Klee–Minty cube in the context of the criss-cross algorith
[3/24] Evaluating: How do the criss-cross algorithm and the simplex algorithm handle the Klee–Minty cube diff
[4/24] Evaluating: How does the Bellman-Ford algorithm handle graphs with negative edge weights?
[5/24] Evaluating: What is the relationship between the Bellman-Ford algorithm and Dijkstra's algorithm?
[6/24] Evaluating: Who contributed to the development of the Bellman-Ford algorithm, and how is it named?
[7/24] Evaluating: How did the publication of 'Artificial Intelligence: A Modern Approach' influence the fiel
[8/24] Evaluating: What is the relationship between the topics covered in AIMA and the development of AI?
[9/24] Evaluating: How has the GitHub repository for AIMA contributed to the learning experien

# Results

In [16]:
# ── Step: Summary metrics ───────────────────────────────────────────────────
metrics = ["faithfulness", "answer_relevance", "context_relevance"]

if not eval_results:
    print("No evaluation results found.")
else:
    averages = {
        m: sum(r[m] for r in eval_results) / len(eval_results)
        for m in metrics
    }
    overall = sum(averages.values()) / len(averages)

    print(f"\n{'='*45}")
    print(f"  RAG Evaluation — dataset: {DATASET_NAME}")
    print(f"{'='*45}")
    print(f"  Questions evaluated : {len(eval_results)}")
    print(f"  Faithfulness        : {averages['faithfulness']:.2f} / 5")
    print(f"  Answer Relevance    : {averages['answer_relevance']:.2f} / 5")
    print(f"  Context Relevance   : {averages['context_relevance']:.2f} / 5")
    print(f"  Overall             : {overall:.2f} / 5")
    print(f"{'='*45}")

    print("\nPer-question scores:")
    print(f"{'#':<4} {'F':>3} {'A':>3} {'C':>3}  Question")
    print("-" * 80)
    for i, r in enumerate(eval_results, 1):
        q_preview = r["question"][:55] + "..." if len(r["question"]) > 55 else r["question"]
        print(f"{i:<4} {r['faithfulness']:>3} {r['answer_relevance']:>3} {r['context_relevance']:>3}  {q_preview}")


  RAG Evaluation — dataset: wiki
  Questions evaluated : 24
  Faithfulness        : 4.75 / 5
  Answer Relevance    : 4.92 / 5
  Context Relevance   : 4.83 / 5
  Overall             : 4.83 / 5

Per-question scores:
#      F   A   C  Question
--------------------------------------------------------------------------------
1      5   5   5  How does the criss-cross algorithm differ from the simp...
2      5   5   5  What is the significance of the Klee–Minty cube in the ...
3      3   3   5  How do the criss-cross algorithm and the simplex algori...
4      5   5   5  How does the Bellman-Ford algorithm handle graphs with ...
5      5   5   5  What is the relationship between the Bellman-Ford algor...
6      5   5   5  Who contributed to the development of the Bellman-Ford ...
7      5   5   5  How did the publication of 'Artificial Intelligence: A ...
8      5   5   5  What is the relationship between the topics covered in ...
9      5   5   5  How has the GitHub repository for AIMA cont